# 🛰️ DisasterLens AI — Training Notebook

**AI-Powered Disaster Damage Assessment Using Deep Learning**

This notebook trains a **U-Net + ResNet-34** semantic segmentation model on the
[xBD dataset](https://xview2.org) to classify building damage into 5 categories:
Background, No Damage, Minor Damage, Major Damage, Destroyed.

---

## Pre-flight checklist
1. ✅ **Runtime → Change runtime type → T4 GPU** (required for training)
2. ✅ **Download xBD** from https://xview2.org (requires registration)
3. ✅ **Upload dataset** to Google Drive at `/MyDrive/xBD/`
4. ✅ **Clone or upload** this repository to `/content/DisasterLens-AI/`

---
⚠️ **Disclaimer:** Predictions require human expert verification.

## 1. Environment Setup

In [1]:
# Colab already ships a CUDA-matched torch / torchvision / opencv - do NOT reinstall them.
!pip install -q segmentation-models-pytorch albumentations shapely reportlab PyYAML
print('✅ Dependencies installed')


✅ Dependencies installed



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch version : 2.14.1+cpu
CUDA available  : False


## 2. Mount Google Drive & Clone Repository

In [4]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os, zipfile

REPO_DIR = '/content/DisasterLens-AI'

# Option A (recommended): upload DisasterLens-AI.zip to  MyDrive/DisasterLens/  once.
ZIP_ON_DRIVE = '/content/drive/MyDrive/DisasterLens/DisasterLens-AI.zip'

# Option B: clone from GitHub instead (set your URL, leave '' to skip)
REPO_URL = ''

if os.path.exists(os.path.join(REPO_DIR, 'config.yaml')):
    print('✅ Project already present')
elif os.path.exists(ZIP_ON_DRIVE):
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_ON_DRIVE) as z:
        names = z.namelist()
        # tolerate a zip whose files sit inside a single top-level folder
        top = names[0].split('/')[0] + '/'
        strip = top if all(n.startswith(top) for n in names) and not names[0].startswith('config.yaml') else ''
        for m in z.infolist():
            rel = m.filename[len(strip):] if strip else m.filename
            if not rel or rel.startswith(('.venv', '__pycache__')):
                continue
            target = os.path.join(REPO_DIR, rel)
            if m.is_dir():
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with open(target, 'wb') as f:
                    f.write(z.read(m))
    print(f'✅ Extracted project to {REPO_DIR}')
elif REPO_URL:
    !git clone {REPO_URL} {REPO_DIR}
else:
    raise FileNotFoundError(
        f'Project not found. Upload DisasterLens-AI.zip to {ZIP_ON_DRIVE} or set REPO_URL.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')


## 3. Configure Dataset & Output Paths

Edit the paths below to match where you uploaded the xBD dataset and where
you want checkpoints saved.

In [ ]:
import yaml
from pathlib import Path

# ─── EDIT THESE PATHS ────────────────────────────────────────────────────────
DATASET_ROOT   = '/content/drive/MyDrive/xBD'          # xBD dataset root
CHECKPOINT_DIR = '/content/drive/MyDrive/DisasterLens/checkpoints'
ARTIFACTS_DIR  = '/content/drive/MyDrive/DisasterLens/artifacts'
# ─────────────────────────────────────────────────────────────────────────────

# Verify dataset exists
ds_path = Path(DATASET_ROOT)
assert ds_path.exists(), f'Dataset not found: {DATASET_ROOT}\nDownload from https://xview2.org'

# Create output directories
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

# Patch config.yaml with our paths
config_path = Path('config.yaml')
with open(config_path) as f:
    config = yaml.safe_load(f)

config['paths']['dataset_root']            = DATASET_ROOT
config['paths']['checkpoint_dir']          = CHECKPOINT_DIR
config['paths']['artifacts_dir']           = ARTIFACTS_DIR
config['paths']['training_history_file']   = str(Path(ARTIFACTS_DIR) / 'training_history.json')
config['paths']['evaluation_results_file'] = str(Path(ARTIFACTS_DIR) / 'evaluation_results.json')
config['training']['num_workers']          = 2  # Colab has 2 CPU cores

# Write patched config
COLAB_CONFIG = '/content/DisasterLens-AI/config_colab.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('✅ Paths configured')
print(f'   Dataset root  : {DATASET_ROOT}')
print(f'   Checkpoints   : {CHECKPOINT_DIR}')
print(f'   Artifacts     : {ARTIFACTS_DIR}')
print(f'   Colab config  : {COLAB_CONFIG}')

## 4. Inspect Dataset

Verify that the dataset loader can discover pairs correctly.

In [ ]:
import sys
sys.path.insert(0, '/content/DisasterLens-AI')

from src.data.dataset import _discover_pairs, split_pairs_by_event

pairs = _discover_pairs(Path(DATASET_ROOT))
print(f'Total image pairs : {len(pairs):,}')

# Show disaster event breakdown
from collections import Counter
event_counts = Counter(p['disaster'] for p in pairs)
print(f'Disaster events   : {len(event_counts)}')
print()
for event, count in sorted(event_counts.items(), key=lambda x: -x[1])[:10]:
    print(f'  {event:<35} {count:>5} pairs')

# Split
tc = config.get('training', {})
train_pairs, val_pairs, test_pairs = split_pairs_by_event(
    pairs,
    train_frac = tc.get('train_split', 0.70),
    val_frac   = tc.get('val_split',   0.15),
    seed       = tc.get('seed',        42),
)
print(f'\nSplit: {len(train_pairs)} train / {len(val_pairs)} val / {len(test_pairs)} test')

In [ ]:
# Visualise a few training samples
import matplotlib.pyplot as plt
import numpy as np
from src.data.dataset import XBDDataset
from src.visualization.overlay import colorize_mask, CLASS_NAMES

sample_ds = XBDDataset(train_pairs[:8], image_size=256, transform=None)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Sample Training Pairs (Post-Disaster Image + Damage Mask)', fontsize=14)

for i in range(min(4, len(sample_ds))):
    item = sample_ds[i]
    post_img = item['image'][:, :, 3:].astype(np.uint8)   # post channels
    mask     = item['mask']
    colored  = colorize_mask(mask)

    axes[i][0].imshow(post_img)
    axes[i][0].set_title(f"{item['metadata']['disaster']}", fontsize=8)
    axes[i][0].axis('off')

    axes[i][1].imshow(colored)
    axes[i][1].set_title('Damage Mask', fontsize=8)
    axes[i][1].axis('off')

    # Pixel distribution
    unique, counts = np.unique(mask[mask != 255], return_counts=True)
    axes[i][2].bar([CLASS_NAMES.get(u,'?') for u in unique], counts,
                   color=['#2D2D2D','#00C851','#FFD600','#FF6D00','#FF1744'][:len(unique)])
    axes[i][2].set_title('Class pixels', fontsize=8)
    axes[i][2].tick_params(axis='x', labelsize=6)

    axes[i][3].axis('off')  # spare column

plt.tight_layout()
plt.show()

## 5. Train the Model

Training runs for the number of epochs configured in `config.yaml` (default 50).
**On a Colab T4 GPU**, expect ~8–15 min/epoch depending on dataset size.

Checkpoints are saved to Google Drive automatically.

### 5a. (Optional) Smoke test first
Runs a single batch to catch path / memory problems in ~1 minute, before committing hours of GPU time. Nothing is saved.

In [ ]:
!python training/train.py --config {COLAB_CONFIG} --dry-run

In [ ]:
# Full training run. Safe to re-run: see the resume cell below if Colab disconnects.
!python training/train.py \
    --config {COLAB_CONFIG} \
    --epochs 50


### 5b. Resume after a disconnect
Training writes `last_checkpoint.pth` to your Drive every epoch. If the runtime is recycled, re-run sections 1-3, then run this cell to continue where you stopped (use the same `--epochs`).

In [ ]:
!python training/train.py \
    --config {COLAB_CONFIG} \
    --epochs 50 \
    --resume {CHECKPOINT_DIR}/last_checkpoint.pth

In [ ]:
# Plot training curves from saved history
import json
import matplotlib.pyplot as plt

history_path = Path(ARTIFACTS_DIR) / 'training_history.json'
assert history_path.exists(), 'training_history.json not found — run the training cell first'

with open(history_path) as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_loss'], label='Train Loss', color='#00D4FF')
ax1.plot(epochs, history['val_loss'],   label='Val Loss',   color='#FF6B6B', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss Curves')

ax2.plot(epochs, history['val_mean_iou'], color='#00FF9F')
ax2.fill_between(epochs, 0, history['val_mean_iou'], alpha=0.1, color='#00FF9F')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Mean IoU'); ax2.set_title('Validation Mean IoU')

plt.tight_layout()
plt.show()
print(f"Best val_mIoU: {max(history['val_mean_iou']):.4f}  "
      f"(epoch {history['val_mean_iou'].index(max(history['val_mean_iou'])) + 1})")

## 6. Evaluate on Test Set

In [ ]:
BEST_CHECKPOINT = str(Path(CHECKPOINT_DIR) / 'best_model.pth')

!python training/evaluate.py \
    --config   {COLAB_CONFIG} \
    --checkpoint {BEST_CHECKPOINT} \
    --split    test

In [ ]:
# Display evaluation results
eval_path = Path(ARTIFACTS_DIR) / 'evaluation_results.json'
with open(eval_path) as f:
    results = json.load(f)

def fmt(v, spec='.4f'):
    return format(v, spec) if isinstance(v, (int, float)) else 'N/A'

print('── Evaluation Summary ─────────────────────────────────────')
print(f"  Pixel Accuracy : {fmt(results.get('accuracy'))}")
print(f"  Mean IoU       : {fmt(results.get('mean_iou'))}")
print(f"  Mean F1        : {fmt(results.get('mean_f1'))}")
print()
print('── Per-class Metrics ──────────────────────────────────────')
print(f"{'Class':<18} {'IoU':>8} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Support':>10}")
print('-' * 64)
for cls_name, m in results.get('per_class', {}).items():
    print(f"{cls_name:<18} {fmt(m.get('iou')):>8} {fmt(m.get('f1')):>8} "
          f"{fmt(m.get('precision')):>10} {fmt(m.get('recall')):>8} "
          f"{m.get('support', 0):>10,}")


## 7. Download Checkpoint & Artifacts

Download the best checkpoint and artifact JSON files to use in the
local Streamlit application.

In [ ]:
from google.colab import files
import shutil

# Download best checkpoint
best_ckpt = Path(CHECKPOINT_DIR) / 'best_model.pth'
if best_ckpt.exists():
    files.download(str(best_ckpt))
    print(f'✅ Downloading {best_ckpt.name}')
else:
    print('⚠️ best_model.pth not found — run training first')

In [ ]:
# Download artifact JSON files (training history + evaluation results)
for fname in ('training_history.json', 'evaluation_results.json'):
    p = Path(ARTIFACTS_DIR) / fname
    if p.exists():
        files.download(str(p))
        print(f'✅ Downloading {fname}')
    else:
        print(f'⚠️ {fname} not found')

## 8. Next Steps — Use in the Streamlit App

After downloading:

1. **Place the checkpoint** (`best_model.pth`) in a local folder, e.g. `DisasterLens-AI/checkpoints/`
2. **Update `config.yaml`**:
   ```yaml
   paths:
     checkpoint_dir: checkpoints
   inference:
     demo_mode: false   # require the real model (optional)
   ```
3. **Copy the JSON artifacts** into `DisasterLens-AI/artifacts/`:
   - `training_history.json`
   - `evaluation_results.json`
4. **Run the app**:
   ```bash
   streamlit run app.py
   ```
5. The sidebar badge changes from *Demo Mode* to *Model Ready*; **Analytics** shows your real
   curves and metrics, and **Image Analysis** runs genuine inference.

---
> ⚠️ **Disclaimer:** Model predictions require human expert verification
> and must not be used as the sole basis for emergency response decisions.
